# Building Multi-Agent Systems with AG2 and Hugging Face

_Authored by: [Faridun Mirzoev](https://huggingface.co/faridunm)_

In this notebook, we'll build a multi-agent system using [AG2](https://ag2.ai) (formerly AutoGen) — an open-source framework for multi-agent AI with 500K+ monthly PyPI downloads and 4,300+ GitHub stars.

We'll connect AG2 agents to Hugging Face models via the Inference API, creating a team of specialized agents that collaborate through AG2's GroupChat to analyze and summarize research topics.

## What you'll learn

- How to configure AG2 to use Hugging Face models via OpenAI-compatible API
- How to create specialized agents with different roles
- How to register Python tools for agents to call
- How to orchestrate multi-agent collaboration with GroupChat

## Setup

Install the required packages:

In [ ]:
!pip install "ag2[openai]>=0.11.4,<1.0" huggingface_hub -q

## Configuring AG2 with Hugging Face Inference API

AG2 can connect to any OpenAI-compatible API endpoint. The Hugging Face Inference API exposes models through such an endpoint, making integration straightforward.

You'll need a [Hugging Face token](https://huggingface.co/settings/tokens) with Inference API access.

In [ ]:
import os
from huggingface_hub import get_token

from autogen import (
    AssistantAgent,
    UserProxyAgent,
    GroupChat,
    GroupChatManager,
    LLMConfig,
)

# Use HF token — set HF_TOKEN env var or login via huggingface-cli
hf_token = get_token()

# Configure AG2 to use a Hugging Face model via the Inference API
# The Inference API provides an OpenAI-compatible endpoint
llm_config = LLMConfig(
    {
        "model": "Qwen/Qwen2.5-Coder-32B-Instruct",
        "api_key": hf_token,
        "api_type": "openai",
        "base_url": "https://router.huggingface.co/v1",
    }
)

## Part 1: Basic Two-Agent Conversation

Let's start with the simplest AG2 pattern — an assistant agent powered by a Hugging Face model talking to a user proxy.

In [ ]:
assistant = AssistantAgent(
    name="Assistant",
    system_message=(
        "You are a helpful AI assistant. "
        "Provide clear, concise answers. "
        "Reply TERMINATE when the task is complete."
    ),
    llm_config=llm_config,
)

user_proxy = UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    is_termination_msg=lambda x: (x.get("content") or "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)

response = user_proxy.run(
    assistant,
    message="What are the key advantages of open-source language models compared to proprietary ones? Give 3 points.",
)
response.process()

Let's see the agent's response:

In [ ]:
for msg in response.messages:
    print(f"**{msg.get('role', 'unknown')}**: {str(msg.get('content', ''))[:500]}")
    print("---")

## Part 2: Adding Tools

AG2 uses a decorator pattern for tool registration. This lets agents call Python functions during the conversation — the model decides when and how to call them.

In [ ]:
from typing import Annotated
from huggingface_hub import HfApi

# Create fresh agents for this example
tool_assistant = AssistantAgent(
    name="HF_Assistant",
    system_message=(
        "You are an assistant that helps explore Hugging Face models and datasets. "
        "Use the available tools to find information. "
        "Reply TERMINATE when done."
    ),
    llm_config=llm_config,
)

tool_user = UserProxyAgent(
    name="Tool_User",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    is_termination_msg=lambda x: (x.get("content") or "").rstrip().endswith("TERMINATE"),
    code_execution_config=False,
)


# Register tools using AG2's decorator pattern
@tool_user.register_for_execution()
@tool_assistant.register_for_llm(description="Search Hugging Face Hub for models matching a query")
def search_models(
    query: Annotated[str, "Search query for models"],
    limit: Annotated[int, "Maximum number of results"] = 5,
) -> str:
    """Search the Hugging Face Hub for models."""
    api = HfApi()
    models = api.list_models(search=query, sort="downloads", limit=limit)
    results = []
    for model in models:
        results.append(f"- **{model.id}** (downloads: {model.downloads:,})")
    return "\n".join(results) if results else "No models found."


@tool_user.register_for_execution()
@tool_assistant.register_for_llm(description="Get details about a specific model on Hugging Face Hub")
def get_model_info(
    model_id: Annotated[str, "The model ID (e.g., 'meta-llama/Llama-3-8B')"],
) -> str:
    """Get detailed information about a model."""
    api = HfApi()
    try:
        info = api.model_info(model_id)
        return (
            f"Model: {info.id}\n"
            f"Downloads (last month): {info.downloads:,}\n"
            f"Likes: {info.likes:,}\n"
            f"Pipeline: {info.pipeline_tag}\n"
            f"Tags: {', '.join(info.tags[:10])}"
        )
    except Exception as e:
        return f"Error: {e}"

In [ ]:
response = tool_user.run(
    tool_assistant,
    message="Find the top 3 most downloaded text-generation models on Hugging Face and tell me about the most popular one.",
)
response.process()

## Part 3: Multi-Agent GroupChat

Now let's use AG2's signature feature — GroupChat. We'll create a team of specialized agents that collaborate to produce a research overview:

- **Researcher**: Gathers facts and key points
- **Writer**: Creates polished content
- **Critic**: Reviews and provides feedback

The GroupChatManager (also powered by the HF model) automatically decides which agent should speak next.

In [ ]:
researcher = AssistantAgent(
    name="Researcher",
    system_message=(
        "You are a research specialist focused on AI and machine learning. "
        "When given a topic, provide key facts, recent developments, and data points. "
        "Be thorough but concise. Present findings as structured bullet points."
    ),
    llm_config=llm_config,
)

writer = AssistantAgent(
    name="Writer",
    system_message=(
        "You are a technical writer. Take research findings and craft a clear, "
        "well-structured summary suitable for a blog post. Keep it under 300 words. "
        "Focus on making complex topics accessible."
    ),
    llm_config=llm_config,
)

critic = AssistantAgent(
    name="Critic",
    system_message=(
        "You are a quality reviewer for technical content. "
        "Evaluate the written content for accuracy, clarity, and completeness. "
        "Provide specific, actionable feedback. "
        "If the content meets high quality standards, reply TERMINATE."
    ),
    llm_config=llm_config,
)

coordinator = UserProxyAgent(
    name="Coordinator",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
)

In [ ]:
group_chat = GroupChat(
    agents=[coordinator, researcher, writer, critic],
    messages=[],
    max_round=8,
    speaker_selection_method="auto",
)

manager = GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
)

response = coordinator.run(
    manager,
    message=(
        "Create a brief overview of the current state of open-source "
        "large language models in 2025-2026. Cover key models, trends, "
        "and what makes open-source LLMs competitive."
    ),
)
response.process()

## Understanding the Conversation Flow

Let's see how the agents collaborated:

In [ ]:
for i, msg in enumerate(response.messages):
    speaker = msg.get("name", msg.get("role", "unknown"))
    content = str(msg.get("content", ""))
    print(f"[{i+1}] {speaker}:")
    print(f"  {content[:300]}...")
    print()

## Key Takeaways

- **AG2 + Hugging Face** works seamlessly via the OpenAI-compatible Inference API — just set `base_url` and use your HF token as the API key
- **Tool registration** uses a clean decorator pattern (`@register_for_execution` + `@register_for_llm`) that lets agents call any Python function
- **GroupChat** orchestrates multiple agents automatically — the manager model decides who speaks next based on conversation context
- **Open-source models** like Qwen, Llama, and Mistral work well as the backbone for AG2 agents

## Next Steps

- Try different Hugging Face models — swap the model ID in `LLMConfig`
- Add more tools (web search, database queries, file operations)
- Experiment with `speaker_selection_method="round_robin"` for predictable turns
- Explore [AG2 documentation](https://docs.ag2.ai) for advanced patterns like nested chats and custom agent types
- Check out other cookbook recipes for [multi-agent RAG](multiagent_rag_system) and [data analysis agents](agent_data_analyst)